# **Spotify Dataset — Data Engineer**

## **Import Libraries**

In [52]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import StandardScaler
from pathlib import Path

## **Project path Configuration**

In [53]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw_data.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw_data.csv"

## **Load the Dataset**

In [54]:
df = pd.read_csv(RAW_DATA_PATH)

### Basic Dataset Check

In [55]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        114000 non-null  int64  
 1   track_id          114000 non-null  str    
 2   artists           113999 non-null  str    
 3   album_name        113999 non-null  str    
 4   track_name        113999 non-null  str    
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   explicit          114000 non-null  bool   
 8   danceability      114000 non-null  float64
 9   energy            114000 non-null  float64
 10  key               114000 non-null  int64  
 11  loudness          114000 non-null  float64
 12  mode              114000 non-null  int64  
 13  speechiness       114000 non-null  float64
 14  acousticness      114000 non-null  float64
 15  instrumentalness  114000 non-null  float64
 16  liveness          114000 non-nu

In [56]:
df.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## **Data Cleaning**

### Remove Unnecessary Index Column

This section removes automatically generated index columns that are not useful for analysis.

In [57]:
df.drop(columns=['Unnamed: 0'], inplace=True)
df.head(1)

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.461,1,-6.746,0,0.143,0.0322,0.000001,0.358,0.715,87.917,4,acoustic


### **Check Missing Values**
This section handles missing values in the dataset.

- Columns with excessive missing values are removed to maintain data quality.  
- After that, rows containing remaining missing values are dropped to ensure consistency for further analysis.

In [58]:
missing_values = df.isnull().sum()

missing_table = missing_values[missing_values > 0].to_frame(name='Missing Count')

print("Columns with missing values:")
print(missing_table)

Columns with missing values:
            Missing Count
artists                 1
album_name              1
track_name              1


### Remove columns with excessive missing values

Columns with a missing value ratio greater than 40% are removed to improve dataset reliability and reduce noise in further analysis.

In [59]:
col_threshold = 0.4    
df = df.loc[:, df.isnull().mean() < col_threshold]
df.shape

(114000, 20)

### Remove remaining Missing values

After removing low-quality columns, rows containing remaining missing values are dropped to ensure dataset consistency.

In [60]:
df = df.dropna()

### Verify missing values after cleaning

This step verifies that the dataset no longer contains missing values after the cleaning process.

In [61]:
df.isnull().sum()

track_id            0
artists             0
album_name          0
track_name          0
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64

### **Duplicate**
This section checks duplicated records in the dataset.

In [62]:
duplicated_rows = df.duplicated().sum()

print(f"Number of duplicated rows: {duplicated_rows}")

Number of duplicated rows: 450


In [63]:
df.drop_duplicates(inplace=True) 

In [64]:
df.reset_index(drop=True, inplace=True)

### **Outlier Detection**

This section identifies numerical columns for outlier processing using the IQR method.

In [82]:
numeric_columns = df.select_dtypes(include=np.number).columns
numeric_columns

Index(['popularity', 'explicit', 'energy', 'key', 'mode', 'speechiness',
       'acousticness', 'instrumentalness', 'liveness', 'valence',
       'time_signature', 'duration_min', 'loudness_scaled'],
      dtype='str')

### Remove Outliers Using IQR

In [83]:
for col in numeric_columns:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = (Q1 - 1.5 * IQR)
    upper_bound = (Q3 + 1.5 * IQR)

    df = df[(df[col] >= lower_bound)&(df[col] <= upper_bound)]

df.shape

(44064, 18)

## **Final Dataset Check**

This section verifies the final dataset structure after preprocessing and cleaning.

In [84]:
df.info()

<class 'pandas.DataFrame'>
Index: 44064 entries, 2 to 113548
Data columns (total 18 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   popularity                     44064 non-null  int64  
 1   explicit                       44064 non-null  int64  
 2   energy                         44064 non-null  float64
 3   key                            44064 non-null  int64  
 4   mode                           44064 non-null  int64  
 5   speechiness                    44064 non-null  float64
 6   acousticness                   44064 non-null  float64
 7   instrumentalness               44064 non-null  float64
 8   liveness                       44064 non-null  float64
 9   valence                        44064 non-null  float64
 10  time_signature                 44064 non-null  int64  
 11  duration_min                   44064 non-null  float64
 12  loudness_scaled                44064 non-null  float64
 13  t

In [85]:
df.isnull().sum()

popularity                       0
explicit                         0
energy                           0
key                              0
mode                             0
speechiness                      0
acousticness                     0
instrumentalness                 0
liveness                         0
valence                          0
time_signature                   0
duration_min                     0
loudness_scaled                  0
tempo_category_medium            0
tempo_category_slow              0
mood_category_Happy/Energetic    0
mood_category_Relaxing           0
mood_category_Sad/Chill          0
dtype: int64

In [86]:
df.head()

,popularity,explicit,energy,key,mode,speechiness,acousticness,instrumentalness,liveness,valence,time_signature,duration_min,loudness_scaled,tempo_category_medium,tempo_category_slow,mood_category_Happy/Energetic,mood_category_Relaxing,mood_category_Sad/Chill
2,57,0,0.359,0,1,0.0557,0.210,0.0,0.1170,0.120,4,3.513767,-0.297440,False,True,False,False,True
4,82,0,0.443,2,1,0.0526,0.469,0.0,0.0829,0.167,4,3.314217,-0.286864,True,False,False,True,False
5,58,0,0.481,6,1,0.1050,0.289,0.0,0.1890,0.666,4,3.570667,-0.112462,True,False,False,True,False
7,80,0,0.444,11,1,0.0417,0.559,0.0,0.0973,0.712,4,4.049100,-0.217024,False,False,False,True,False
8,74,0,0.414,0,1,0.0369,0.294,0.0,0.1510,0.669,4,3.160217,-0.091111,True,False,False,True,False


## Export Cleaned Dataset

In [87]:
df.to_csv('cleaned_data.csv', index=False)
print("Cleaned dataset exported successfully!")

Cleaned dataset exported successfully!
